# Урок 3.3 — Фильтрация, агрегации и даты: заказы во времени (DataFrame API + Spark SQL)

Этот ноутбук — шаблон урока 3.3 модуля «Основы Spark» для стенда `spark_01`.

Здесь мы закрепляем базовые навыки, которые **нужны перед JOIN-уроком 3.4**:

---

## Что вы сделаете

1) Научитесь уверенно **фильтровать** данные по статусам и датам.
2) Разберёте **работу с датами и временем**: `to_date`, `date_trunc`, `year`, `month`, `datediff`.
3) Сделаете **агрегации**: по дням, по месяцам, по статусам.
4) Соберёте метрику **суммы заказа** из `order_items` через группировку.
5) Сделаете **безопасный 1:1 JOIN после агрегации** (подготовка к 3.4).
6) Идемпотентно запишете результаты в Parquet и прочитаете обратно.
7) Выполните 5 практических заданий + запустите ✅ проверки.

---

## Правило путей в стенде

- исходные данные читаем из `/data/csv`
- результаты пишем в `/workspace/samples/lesson03_03/...`

Причина: и driver в Jupyter, и executors на воркерах должны видеть **одинаковые пути**.

Важно: любую ячейку можно перезапускать — DataFrame просто переприсвоится.

---
## 0. Импорты и SparkSession

В стенде `spark_01` Spark-сессия обычно уже доступна как `spark`.
Ниже — безопасная проверка на случай автономного запуска ноутбука.


In [ ]:
# Импорты + безопасное получение SparkSession
from pathlib import Path
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

try:
    spark  # noqa: F821
except NameError:
    spark = SparkSession.builder.getOrCreate()

# Для локального стенда часто полезно уменьшить число шифлов (нагляднее и быстрее)
spark.conf.set("spark.sql.shuffle.partitions", "8")

spark

---
## 1. Пути и хелперы

Заведём утилиты, чтобы:
- одинаково читать CSV,
- одинаково делать идемпотентную запись,
- быстро считать строки и уникальные ключи.

Важно: Spark почти всегда пишет **в директорию**, а не в один файл.


In [ ]:
# Пути /data и /workspace, плюс хелперы для чтения/записи
DATA_DIR = "/data/csv"
WORKSPACE_DIR = "/workspace"
SAMPLES_DIR = f"{WORKSPACE_DIR}/samples"
# LESSON_DIR = f"{SAMPLES_DIR}/lesson03_03"
LESSON_DIR = f"{WORKSPACE_DIR}/lesson03_03"

os.makedirs(LESSON_DIR, exist_ok=True)

ORDERS_CSV = f"{DATA_DIR}/olist_orders_dataset.csv"
ORDER_ITEMS_CSV = f"{DATA_DIR}/olist_order_items_dataset.csv"

OUT_EXAMPLE_PARQUET = f"{LESSON_DIR}/orders_daily_status_parquet"

def read_csv(path: str):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

def reset_dir(path: str) -> None:
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

def write_parquet_dir(df, path: str, single_file: bool = True):
    reset_dir(path)
    writer_df = df.coalesce(1) if single_file else df
    (
        writer_df.write
        .mode("overwrite")
        .parquet(path)
    )
    return os.listdir(path)

def read_parquet_dir(path: str):
    return spark.read.parquet(path)

def agg_counts(df, key_col: str = None):
    if key_col is None:
        return int(df.agg(F.count("*").alias("rows")).first()["rows"])

    row = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(key_col).alias(f"distinct_{key_col}")
    ).first()
    return int(row["rows"]), int(row[f"distinct_{key_col}"])

(LESSON_DIR, ORDERS_CSV, ORDER_ITEMS_CSV)

---
## 2. Читаем таблицы Olist

Читаем:
- `orders` — заказы
- `order_items` — позиции заказов

Для обучения используем `inferSchema=True`.


In [ ]:
# Читаем исходные CSV в DataFrame
df_orders_raw = read_csv(ORDERS_CSV)
df_order_items_raw = read_csv(ORDER_ITEMS_CSV)

# Для урока можно закешировать (мы будем делать несколько действий)
df_orders_raw.cache(); df_order_items_raw.cache()

# Материализация кэша одним действием
_ = df_orders_raw.count()

df_orders_raw, df_order_items_raw

---
## 3. Контракт таблиц + базовая работа с датами

Сужаем таблицы до нужных колонок, чтобы:
- меньше таскать данных,
- не ловить коллизии имён,
- держать результат читаемым.

Также добавим поля:
- `order_purchase_date` — дата покупки (без времени)
- `order_purchase_month` — месяц покупки (через `date_trunc('month', ...)`)


In [ ]:
# Контракты: выбираем только нужные колонки и добавляем дату/месяц
df_orders = (
    df_orders_raw.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    )
    .withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))
    .withColumn("order_purchase_month", F.date_trunc("month", F.col("order_purchase_timestamp")))
)

df_order_items = df_order_items_raw.select(
    "order_id",
    "order_item_id",
    "product_id",
    "price",
    "freight_value"
)

df_orders.show(5, truncate=False)
df_order_items.show(5, truncate=False)

### Мини-сводка: rows + distinct ключей

Проверим базовые ожидания:
- `orders`: 1 строка = 1 заказ (`order_id` уникален)
- `order_items`: много строк на один `order_id`


In [ ]:
# Быстрая сводка по количествам
orders_rows, orders_dist = agg_counts(df_orders, "order_id")
items_rows, items_orders_dist = agg_counts(df_order_items, "order_id")

{
  "orders": {"rows": orders_rows, "distinct_order_id": orders_dist},
  "order_items": {"rows": items_rows, "distinct_order_id": items_orders_dist}
}

---
## 4. Фильтрация: статусы и диапазоны дат

Тренируем базовые фильтры:
- по статусу (`order_status`)
- по дате покупки (`order_purchase_date`)

Обратите внимание: для дат удобно фильтровать по `order_purchase_date` (без времени).


In [ ]:
# Примеры фильтров: delivered + диапазон дат
df_delivered = df_orders.filter(F.col("order_status") == "delivered")

# Диапазон дат (пример): 2017-01-01 .. 2017-01-31
df_jan_2017 = df_orders.filter(
    (F.col("order_purchase_date") >= F.lit("2017-01-01"))
    & (F.col("order_purchase_date") <= F.lit("2017-01-31"))
)

df_delivered.select("order_id", "order_status", "order_purchase_date").show(5, truncate=False)
df_jan_2017.select("order_id", "order_status", "order_purchase_date").show(5, truncate=False)

(df_orders.count(), df_delivered.count(), df_jan_2017.count())

---
## 5. Агрегации во времени

Соберём 2 базовые витрины:

1) **Заказы по дням**
- `order_purchase_date`
- `orders_cnt`

2) **Заказы по месяцам и статусам**
- `order_purchase_month`
- `order_status`
- `orders_cnt`


In [ ]:
# Агрегация 1: заказы по дням
df_orders_daily = (
    df_orders
    .groupBy("order_purchase_date")
    .agg(F.count("*").alias("orders_cnt"))
    .orderBy("order_purchase_date")
)

df_orders_daily.show(10, truncate=False)
df_orders_daily.count()

In [ ]:
# Агрегация 2: заказы по месяцам и статусам
df_orders_month_status = (
    df_orders
    .groupBy("order_purchase_month", "order_status")
    .agg(F.count("*").alias("orders_cnt"))
    .orderBy("order_purchase_month", "order_status")
)

df_orders_month_status.show(20, truncate=False)
df_orders_month_status.count()

---
## 6. Сумма заказа из order_items (без JOIN)

Соберём стоимость заказа из позиций:

- `item_total_value = price + freight_value`
- `order_total_value = sum(item_total_value)` по `order_id`

Это важный паттерн: сначала **агрегируем до grain заказа**, а уже потом (в следующем шаге) делаем безопасный join.


In [ ]:
# Считаем сумму заказа из order_items
df_order_totals = (
    df_order_items
    .withColumn("item_total_value", F.col("price") + F.col("freight_value"))
    .groupBy("order_id")
    .agg(F.sum(F.col("item_total_value").cast("double")).alias("order_total_value"))
)

# order_id тут должен быть уникальным
df_order_totals.show(10, truncate=False)
df_order_totals.count()

---
## 7. Мини-превью JOIN: безопасный 1:1 после агрегации

Мы ещё не изучаем «опасные» многотабличные JOIN (это урок 3.4).

Но здесь важно увидеть правильный подход:
- `orders` имеет grain **1 строка = 1 order_id**
- `order_totals` имеет grain **1 строка = 1 order_id**

Поэтому join по `order_id` — безопасный **1:1**.

Соберём витрину delivered-выручки по дням:
- фильтруем `orders` по `delivered`
- join с `order_totals`
- группируем по `order_purchase_date`


In [ ]:
# Безопасный join 1:1 (orders + order_totals)
df_delivered_daily_revenue = (
    df_orders.filter(F.col("order_status") == "delivered")
    .join(df_order_totals, on="order_id", how="inner")
    .groupBy("order_purchase_date")
    .agg(
        F.countDistinct("order_id").alias("delivered_orders_cnt"),
        F.sum(F.col("order_total_value").cast("double")).alias("revenue_total")
    )
    .orderBy("order_purchase_date")
)

df_delivered_daily_revenue.show(15, truncate=False)
df_delivered_daily_revenue.count()

---
## 8. Тот же результат через Spark SQL

Паттерн:
1) зарегистрировать temp view,
2) написать SQL,
3) сравнить базовые метрики (count, sum).


In [ ]:
# Регистрируем temp views для Spark SQL
df_orders.createOrReplaceTempView("orders")
df_order_items.createOrReplaceTempView("order_items")

In [ ]:
# SQL-версия delivered revenue по дням
query = """
WITH order_totals AS (
  SELECT
    order_id,
    SUM(CAST(price AS double) + CAST(freight_value AS double)) AS order_total_value
  FROM order_items
  GROUP BY order_id
), delivered AS (
  SELECT
    order_id,
    to_date(order_purchase_timestamp) AS order_purchase_date
  FROM orders
  WHERE order_status = 'delivered'
)
SELECT
  d.order_purchase_date,
  COUNT(DISTINCT d.order_id) AS delivered_orders_cnt,
  SUM(t.order_total_value) AS revenue_total
FROM delivered d
JOIN order_totals t
  ON d.order_id = t.order_id
GROUP BY d.order_purchase_date
ORDER BY d.order_purchase_date
"""

df_sql = spark.sql(query)
df_sql.show(15, truncate=False)

(df_delivered_daily_revenue.count(), df_sql.count())

---
## 9. Запись примера в Parquet + read-back check

Идемпотентно:
- чистим путь,
- пишем `overwrite`,
- читаем обратно,
- сравниваем `count()`.


In [ ]:
# Пишем примерный результат урока
files_written = write_parquet_dir(df_delivered_daily_revenue, OUT_EXAMPLE_PARQUET, single_file=True)
files_written

In [ ]:
# Читаем обратно и сравниваем count
df_back = read_parquet_dir(OUT_EXAMPLE_PARQUET)
(df_delivered_daily_revenue.count(), df_back.count())

---
# Практика

Правила выполнения заданий:

1) Не меняйте ячейки выше.
2) Каждое задание решайте в новых ячейках ниже.
3) Для каждого задания:
   - сделайте трансформацию
   - покажите результат `show(...)`
   - запишите результат в указанную папку внутри `/workspace/samples/lesson03_03/...`
   - прочитайте обратно и сравните `count()`

Доступные датафреймы:
- `df_orders`, `df_order_items`, `df_order_totals`


In [ ]:
# Пути для практики (используются и в проверках)
TASK01_OUT = f"{LESSON_DIR}/task01_delivered_orders_parquet"
TASK02_OUT = f"{LESSON_DIR}/task02_orders_daily_parquet"
TASK03_OUT = f"{LESSON_DIR}/task03_month_status_parquet"
TASK04_OUT = f"{LESSON_DIR}/task04_order_totals_parquet"
TASK05_OUT = f"{LESSON_DIR}/task05_delivered_daily_revenue_parquet"

(TASK01_OUT, TASK02_OUT, TASK03_OUT, TASK04_OUT, TASK05_OUT)

## Задание 1. Delivered заказы с датой покупки

Вход:
- DataFrame: `df_orders`

Сделать:
- отфильтровать только `order_status = 'delivered'`

Вывести поля:
- `order_id`
- `customer_id`
- `order_purchase_date`

Записать результат в Parquet:
- `/workspace/samples/lesson03_03/task01_delivered_orders_parquet`

Критерии приёмки:
- `show(10, truncate=False)`
- read-back `count()` совпадает


In [ ]:
df_orders = read_csv(ORDERS_CSV)

df_orders = (
    df_orders\
    .filter(F.col("order_status") == "delivered")
    .withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))
    .select(
        "order_id",
        "customer_id",
        "order_purchase_date"
    )
)

files_written = write_parquet_dir(df_orders, TASK01_OUT)

df_orders.show(10, truncate=False)

df_read = read_parquet_dir(TASK01_OUT)
print(f"Count before: {df_orders.count()}, count after {df_read.count()} ")

## Задание 2. Заказы по дням

Вход:
- DataFrame: `df_orders`

Сделать:
- агрегировать по `order_purchase_date`
- посчитать `orders_cnt`

Вывести поля:
- `order_purchase_date`
- `orders_cnt`

Отсортировать:
- по `order_purchase_date` по возрастанию

Записать результат в Parquet:
- `/workspace/samples/lesson03_03/task02_orders_daily_parquet`

Критерии приёмки:
- `show(20, truncate=False)`
- read-back `count()` совпадает


In [ ]:
df_orders = read_csv(ORDERS_CSV)
df_orders = df_orders.withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))

print(f"Total orders: {df_orders.count()}")

df_orders_daily = (
    df_orders
    .groupBy("order_purchase_date")
    .agg(F.count("*").alias("orders_cnt"))
    .orderBy("order_purchase_date")
)

df_orders_daily.show(20, truncate=False)
files_written = write_parquet_dir(df_orders_daily, TASK02_OUT)

df_read = read_parquet_dir(TASK02_OUT)
print(f"Count before: {df_orders_daily.count()}, count after {df_read.count()} ")

## Задание 3. Заказы по месяцам и статусам

Вход:
- DataFrame: `df_orders`

Сделать:
- агрегировать по `order_purchase_month` и `order_status`
- посчитать `orders_cnt`

Вывести поля:
- `order_purchase_month`
- `order_status`
- `orders_cnt`

Отсортировать:
- по `order_purchase_month` по возрастанию
- затем по `order_status` по возрастанию

Записать результат в Parquet:
- `/workspace/samples/lesson03_03/task03_month_status_parquet`

Критерии приёмки:
- `show(30, truncate=False)`
- read-back `count()` совпадает


In [ ]:
df_orders = read_csv(ORDERS_CSV)
df_orders = df_orders.withColumn("order_purchase_month", F.date_trunc("month", F.col("order_purchase_timestamp")))

df_orders_month_status = (
    df_orders\
    .groupBy("order_purchase_month", "order_status")
    .agg(F.count("*").alias("orders_cnt"))
    .orderBy("order_purchase_month", "order_status")
)

df_orders_month_status.show(20, truncate=False)

files_written = write_parquet_dir(df_orders_month_status, TASK03_OUT)

df_read = read_parquet_dir(TASK03_OUT)
print(f"Count before: {df_orders_month_status.count()}, count after {df_read.count()} ")

## Задание 4. Сумма заказа из order_items

Вход:
- DataFrame: `df_order_items`

Сделать:
- `item_total_value = price + freight_value`
- агрегировать по `order_id`
- `order_total_value = sum(item_total_value)`

Вывести поля:
- `order_id`
- `order_total_value`

Записать результат в Parquet:
- `/workspace/samples/lesson03_03/task04_order_totals_parquet`

Критерии приёмки:
- `order_id` должен быть уникальным
- read-back `count()` совпадает


In [ ]:
df_order_items = read_csv(ORDER_ITEMS_CSV)

df_order_items = df_order_items.withColumn("item_total_value", F.col("price") + F.col("freight_value"))

df_order_totals = (
    df_order_items
    .groupBy("order_id")
    .agg(F.sum("item_total_value").cast("double").alias("order_total_value"))
)

df_order_totals.show(10, truncate=False)
files_written = write_parquet_dir(df_order_totals, TASK04_OUT)

df_read = read_parquet_dir(TASK04_OUT)
print(f"Count before: {df_order_totals.count()}, count after: {df_read.count()}")


## Задание 5. Delivered выручка по дням (безопасный JOIN после агрегации)

Вход:
- `df_orders`
- `df_order_totals` (или результат вашего TASK04)

Сделать:
- оставить только delivered заказы
- join по `order_id` с суммами заказов
- агрегировать по `order_purchase_date`

Вывести поля:
- `order_purchase_date`
- `delivered_orders_cnt` — число уникальных заказов
- `revenue_total` — сумма `order_total_value`

Отсортировать:
- по `order_purchase_date` по возрастанию

Записать результат в Parquet:
- `/workspace/samples/lesson03_03/task05_delivered_daily_revenue_parquet`

Критерии приёмки:
- `show(20, truncate=False)`
- read-back `count()` совпадает


In [ ]:
df_orders = read_csv(ORDERS_CSV)
df_orders = df_orders.withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))

df_order_totals = read_parquet_dir(TASK04_OUT)

df_delivered_daily_revenue = (
    df_orders.filter(F.col("order_status") == "delivered")
    .join(df_order_totals, on="order_id", how="inner")
    .groupBy("order_purchase_date")
    .agg(
        F.countDistinct("order_id").alias("delivered_orders_cnt"),
        F.sum(F.col("order_total_value").cast("double")).alias("revenue_total")
    )
    .orderBy("order_purchase_date")
)

df_delivered_daily_revenue.show(20, truncate=False)

df_order_totals.show(20, truncate=False)
files_written = write_parquet_dir(df_delivered_daily_revenue, TASK05_OUT)

df_read = read_parquet_dir(TASK05_OUT)
print(f"Count before: {df_delivered_daily_revenue.count()}, count after: {df_read.count()}")

---
# ✅ Проверки

Запускайте check-ячейки ниже: получите **OK** или **НЕ OK** + причину.

Важно:
- проверки читают результаты из путей `TASK01_OUT ... TASK05_OUT`
- порядок строк после записи/чтения **не гарантирован**


In [ ]:
# Утилиты для проверок
def ok(task: str, details: str = "") -> None:
    msg = f"OK — {task}"
    if details:
        msg += f" | {details}"
    print(msg)

def bad(task: str, err: Exception) -> None:
    print(f"НЕ OK — {task} | {err}")

def run_check(task: str, check_fn) -> None:
    try:
        check_fn()
        ok(task)
    except AssertionError as e:
        bad(task, e)

def assert_exists(path: str) -> None:
    assert os.path.exists(path), f"путь не найден: {path}"

def read_parquet(path: str):
    return spark.read.parquet(path)

def assert_cols(back_df, expected_cols):
    assert back_df.columns == expected_cols, f"ожидаются колонки: {expected_cols}, получено: {back_df.columns}"

def approx_equal(a: float, b: float, eps: float = 1e-6):
    return abs(a - b) <= eps

def check_duplicates(df, key_cols):
    return (
        df.groupBy(*key_cols)
        .agg(F.count("*").alias("cnt"))
        .filter(F.col("cnt") > 1)
    )

## Проверка задания 1 — delivered orders (Parquet)

In [ ]:
def check_task01():
    assert_exists(TASK01_OUT)
    back = read_parquet(TASK01_OUT)
    assert_cols(back, ["order_id", "customer_id", "order_purchase_date"])

    expected = (
        df_orders
        .filter(F.col("order_status") == "delivered")
        # .select("order_id", "customer_id", "order_purchase_date")
        .select("order_id", "customer_id", "order_purchase_timestamp")
    )

    assert back.count() == expected.count(), "количество delivered заказов не совпало"

run_check("TASK01", check_task01)

## Проверка задания 2 — orders daily (Parquet)

In [ ]:
def check_task02():
    assert_exists(TASK02_OUT)
    back = read_parquet(TASK02_OUT)
    assert_cols(back, ["order_purchase_date", "orders_cnt"])

    expected = (
        df_orders
        .groupBy("order_purchase_date")
        .agg(F.count("*").alias("orders_cnt"))
    )

    assert back.count() == expected.count(), "число дней в агрегации не совпало"

    s_back = back.select(F.sum(F.col("orders_cnt").cast("long")).alias("s")).first()["s"]
    assert int(s_back) == df_orders.count(), "сумма orders_cnt должна равняться числу строк df_orders"

run_check("TASK02", check_task02)

## Проверка задания 3 — month x status (Parquet)

In [ ]:
def check_task03():
    assert_exists(TASK03_OUT)
    back = read_parquet(TASK03_OUT)
    assert_cols(back, ["order_purchase_month", "order_status", "orders_cnt"])

    expected = (
        df_orders
        .groupBy("order_purchase_month", "order_status")
        .agg(F.count("*").alias("orders_cnt"))
    )

    assert back.count() == expected.count(), "число групп (month,status) не совпало"

    s_back = back.select(F.sum(F.col("orders_cnt").cast("long")).alias("s")).first()["s"]
    assert int(s_back) == df_orders.count(), "сумма orders_cnt должна равняться числу строк df_orders"

run_check("TASK03", check_task03)

## Проверка задания 4 — order totals (Parquet)

In [ ]:
def check_task04():
    assert_exists(TASK04_OUT)
    back = read_parquet(TASK04_OUT)
    assert_cols(back, ["order_id", "order_total_value"])

    # order_id должен быть уникальным
    dups = check_duplicates(back, ["order_id"]).count()
    assert dups == 0, "order_id должен быть уникальным"

    expected = (
        df_order_items
        .withColumn("item_total_value", F.col("price") + F.col("freight_value"))
        .groupBy("order_id")
        .agg(F.sum(F.col("item_total_value").cast("double")).alias("order_total_value"))
    )

    assert back.count() == expected.count(), "число заказов (order_id) не совпало"

    sum_back = back.select(F.sum(F.col("order_total_value").cast("double")).alias("s")).first()["s"]
    sum_exp = expected.select(F.sum(F.col("order_total_value").cast("double")).alias("s")).first()["s"]
    sum_back = float(sum_back) if sum_back is not None else 0.0
    sum_exp = float(sum_exp) if sum_exp is not None else 0.0
    assert approx_equal(sum_back, sum_exp, eps=1e-5), "сумма order_total_value не совпадает с эталоном"

run_check("TASK04", check_task04)

## Проверка задания 5 — delivered daily revenue (Parquet)

In [ ]:
def check_task05():
    assert_exists(TASK05_OUT)
    back = read_parquet(TASK05_OUT)
    assert_cols(back, ["order_purchase_date", "delivered_orders_cnt", "revenue_total"])

    expected = (
        df_orders.filter(F.col("order_status") == "delivered")
        .join(df_order_totals, on="order_id", how="inner")
        .groupBy("order_purchase_date")
        .agg(
            F.countDistinct("order_id").alias("delivered_orders_cnt"),
            F.sum(F.col("order_total_value").cast("double")).alias("revenue_total")
        )
    )

    assert back.count() == expected.count(), "число дней в delivered revenue не совпало"

    sum_back = back.select(F.sum(F.col("revenue_total").cast("double")).alias("s")).first()["s"]
    sum_exp = expected.select(F.sum(F.col("revenue_total").cast("double")).alias("s")).first()["s"]
    sum_back = float(sum_back) if sum_back is not None else 0.0
    sum_exp = float(sum_exp) if sum_exp is not None else 0.0
    assert approx_equal(sum_back, sum_exp, eps=1e-4), "сумма revenue_total не совпадает с эталоном"

run_check("TASK05", check_task05)